# **SmoothGrad, VarGrad and a sanity check on the map**

Practice for the module [«Attribution: the gradient as a heuristic»](https://ai-interpretability.school).

The lesson made two claims and left both without code:

1. a Vanilla Gradients map is noisy, and the cure is averaging over noisy copies;
2. before trusting any map, run it through a sanity check — «that is a dozen lines of code
   and one evening».

Here we write those ten lines. By the end of the notebook you will have:

- your own SmoothGrad, SmoothGrad² and VarGrad — no libraries, straight from the formulas;
- a numerical check of the identity $\text{VarGrad} = \text{SmoothGrad}^2 - (\text{SmoothGrad})^2$;
- weight randomization and an answer to the question of whether your map depends on what
  the network has actually learnt.

The notebook runs on CPU in a couple of minutes.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0);   # random noise, but reproducible


## The model and the image

We take the same pair as the rest of the block: ResNet-50 with ImageNet weights and
a photograph from the open course repository.

**`eval()` is a must.** In train mode BatchNorm uses the statistics of a batch of one image,
and the prediction stops matching the real one — and all the attributions drift with it.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

image = Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/hog.jpg').content)).convert('RGB')
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    target = model(x).argmax().item()
print('Class:', target, categories[target])

## 1. Vanilla Gradients — what we are improving

Recall the formula from the lesson: take the gradient of the class logit with respect to the
input and collapse the three channels into one by taking the maximum **of the absolute values**:

$$M_{ij} = \max_k |w_{ijk}|, \qquad w = \frac{\partial F_c(x)}{\partial x}$$

The order of operations matters: absolute value first, maximum second.

In [ ]:
def vanilla_gradient(model, x, target):
    """Gradient of the class logit w.r.t. the input — a tensor shaped like the input."""
    x = x.clone().requires_grad_(True)
    logit = model(x)[0, target]
    grad, = torch.autograd.grad(logit, x)
    return grad

def to_map(grad):
    """Collapsing the channels into a map: absolute value first, maximum second."""
    return grad.abs().max(dim=1).values[0]

vanilla = to_map(vanilla_gradient(model, x, target))
print('map size:', tuple(vanilla.shape))

**Task 1.** Compare the two orders on this very map: `grad.abs().max(1)` and
`grad.max(1).values.abs()`. For how many of the 224×224 pixels do the values differ
by more than 0.001?

In [ ]:
# Your code here

## 2. SmoothGrad and its family

All three quantities come from one set of maps over noisy copies:

$$\text{SmoothGrad} = \frac1n\sum_i M(x+\varepsilon_i), \quad
\text{SmoothGrad}^2 = \frac1n\sum_i M(x+\varepsilon_i)^2, \quad
\text{VarGrad} = \text{SmoothGrad}^2 - (\text{SmoothGrad})^2$$

The noise level is set as a fraction of the range of the input:
$\sigma = \text{fraction}\times(x_{max}-x_{min})$.

In [ ]:
def noisy_maps(model, x, target, sigma_frac=0.15, n=25):
    """n maps over noisy copies of the input. Returns a tensor (n, H, W)."""
    sigma = sigma_frac * (x.max() - x.min())
    maps = []
    for _ in range(n):
        noisy = x + torch.randn_like(x) * sigma
        maps.append(to_map(vanilla_gradient(model, noisy, target)))
    return torch.stack(maps)

maps = noisy_maps(model, x, target)
smoothgrad = maps.mean(0)
smoothgrad_sq = (maps ** 2).mean(0)
vargrad = smoothgrad_sq - smoothgrad ** 2

print('SmoothGrad  mean:', round(smoothgrad.mean().item(), 5))
print('SmoothGrad² mean:', round(smoothgrad_sq.mean().item(), 5))
print('VarGrad     mean:', round(vargrad.mean().item(), 5))

**Task 2.** The lesson says that confusing «the mean of the squares» with «the square of the
mean» is the most common implementation error. Compute both and check that
$\text{SmoothGrad}^2 \geq (\text{SmoothGrad})^2$ pointwise. How many pixels break the
inequality? (The right answer is zero: this is Jensen's inequality.)

In [ ]:
# Your code here

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (m, t) in zip(ax, [(vanilla, 'Vanilla'), (smoothgrad, 'SmoothGrad'),
                          (smoothgrad_sq, 'SmoothGrad²'), (vargrad, 'VarGrad')]):
    a.imshow(m.detach().numpy(), cmap='hot')
    a.set_title(t)
    a.axis('off')
plt.tight_layout()
plt.show()

## 3. The hyperparameter that you choose

The noise level $\sigma$ is not an implementation detail but a choice made by the author of the
explanation: it moves the map across a wide range, and the range differs for every
«image — model» pair.

In [ ]:
for frac in (0.05, 0.15, 0.35):
    m = noisy_maps(model, x, target, sigma_frac=frac, n=15).mean(0)
    # доля «массы» карты, попавшая в 5 % самых ярких пикселей
    top = torch.topk(m.flatten(), k=m.numel() // 20).values.sum() / m.sum()
    print(f'σ = {frac:.2f} of the range → the 5 % brightest pixels hold {top:.1%} of the mass of the map')

**Task 3.** What happens to the concentration of the map as the noise grows, and why?
Write the answer in two lines.

## 4. Sanity check: does the map depend on what the network learnt

The check from [Adebayo et al., 2018](https://arxiv.org/abs/1810.03292): randomly reinitialize
the weights — first of the last block, then deeper and deeper — and rebuild the map each time.
If the map does not change, it is not explaining the model.

For similarity we take something simple: the Spearman correlation between the maps flattened
into vectors.

In [ ]:
from scipy.stats import spearmanr

def spearman(a, b):
    return spearmanr(a.flatten().detach().numpy(), b.flatten().detach().numpy()).statistic

def randomize(model, blocks, seed=0):
    """A copy of the network with the listed blocks randomly reinitialized."""
    import copy
    torch.manual_seed(seed)   # random weights, but the same on every run
    broken = copy.deepcopy(model)
    for name in blocks:
        for m in getattr(broken, name).modules():
            if hasattr(m, 'reset_parameters'):
                m.reset_parameters()
    return broken.eval()

cascade = [['fc'], ['fc', 'layer4'], ['fc', 'layer4', 'layer3'], ['fc', 'layer4', 'layer3', 'layer2']]
print(f'{"reinitialized":42s} correlation with the original map')
for blocks in cascade:
    broken = randomize(model, blocks)
    m = to_map(vanilla_gradient(broken, x, target))
    print(f'{", ".join(blocks):42s} {spearman(vanilla, m):+.3f}')

**Task 4.** The correlation should fall as more and more layers are broken. What is the
correlation after reinitializing `fc` and `layer4` (round to hundredths)?

**Task 5.** Do the same for the SmoothGrad map (`n=10` is enough). Does SmoothGrad pass the
check the way Vanilla Gradients does?

In [ ]:
# Your code here

## What to take away

- **SmoothGrad, SmoothGrad² and VarGrad come from one set of maps** — the only difference is
  what you take from the set: the mean, the mean of the squares, or the variance.
- **VarGrad and SmoothGrad² are non-negative by construction**, so the sign of the contribution
  is lost. If «for or against the class» matters to you, take plain SmoothGrad.
- **The noise level is your choice**, and it moves the map noticeably. Report it next to the picture.
- **A sanity check costs ten lines.** Run it on your own task before drawing conclusions:
  a map that does not follow the weights is not explaining the model.

The full discussion of evaluating explanations is in the module on the quality of explanations.